# Stage 4B: Feature Engineering, Encoding & Pipeline Serialization
**Architecture:** Scikit-Learn `ColumnTransformer` integrating:
1. Continuous Numeric Pipeline: Median Imputer + `StandardScaler`
2. High-Cardinality Categorical Pipeline: Mode Imputer + `OneHotEncoder(handle_unknown='ignore')`
3. Road Infrastructure & Temporal Flags: Passthrough binary indicators
**Exported Artifact:** `models/preprocessor.joblib`

1. Data Preparation & Rare Category Binning

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# 1. Load data & run identical initial preparation
df = pd.read_csv("../data/US_Accidents_2016_2023_1M.csv")
df["Precipitation(in)"] = df["Precipitation(in)"].fillna(0.0)

infra_cols = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction", 
    "No_Exit", "Railway", "Roundabout", "Stop", "Traffic_Signal"
]
df[infra_cols] = df[infra_cols].fillna(False).astype(int)

df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")
df["Hour"] = df["Start_Time"].dt.hour
df["DayOfWeek"] = df["Start_Time"].dt.dayofweek
df["Month"] = df["Start_Time"].dt.month
df["Is_Rush_Hour"] = (df["Hour"].between(7, 9) | df["Hour"].between(16, 18)).astype(int)
df = df.drop(columns=["Start_Time"])

# 2. Rare Category Binning (Prevents sparse explosion)
top_weather = df["Weather_Condition"].value_counts().nlargest(20).index
df["Weather_Condition"] = df["Weather_Condition"].apply(
    lambda x: x if x in top_weather or pd.isna(x) else "Other"
)

# 3. Stratified Split
X = df.drop(columns=["Severity"])
y = df["Severity"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Data partitioned and rare weather categories grouped.")

Data partitioned and rare weather categories grouped.


2. ColumnTransformer Construction & Fitting

In [2]:
numeric_features = [
    "Temperature(F)", "Humidity(%)", "Pressure(in)", 
    "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)", 
    "Hour", "DayOfWeek", "Month"
]
categorical_features = ["Weather_Condition", "Sunrise_Sunset", "State"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
    ("pass", "passthrough", infra_cols + ["Is_Rush_Hour"])
])

# Fit PREPROCESSOR STRICTLY ON TRAINING DATA
preprocessor.fit(X_train)

# Transform both folds
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Preprocessed Training Feature Shape: {X_train_processed.shape}")
print(f"Preprocessed Testing Feature Shape:  {X_test_processed.shape}")

Preprocessed Training Feature Shape: (693438, 92)
Preprocessed Testing Feature Shape:  (173360, 92)


3. Pipeline Serialization & Modeling Slice Export

In [3]:
# 1. Export preprocessor pipeline binary
os.makedirs("../models", exist_ok=True)
joblib.dump(preprocessor, "../models/preprocessor.joblib")
print("Saved fitted ColumnTransformer to '../models/preprocessor.joblib'")

# 2. Extract stratified 100k working slice for Stage 6 modeling
train_sample_idx = (
    X_train.groupby(y_train, group_keys=False)
    .apply(lambda s: s.sample(frac=0.125, random_state=42))
    .index
)

X_train_slice = X_train.loc[train_sample_idx]
y_train_slice = y_train.loc[train_sample_idx]

print(f"Working training slice ready for Stage 6: {X_train_slice.shape[0]:,} rows")

Saved fitted ColumnTransformer to '../models/preprocessor.joblib'
Working training slice ready for Stage 6: 86,680 rows


4. Preprocessor Smoke Test & Feature Dimension Inspection

In [4]:
# Production Deserialization & Smoke Test
loaded_prep = joblib.load("../models/preprocessor.joblib")

# Test transformation on a single test record
sample_test_row = X_test.iloc[0:1]
transformed_mock = loaded_prep.transform(sample_test_row)

# Extract generated one-hot feature names
cat_encoder = loaded_prep.named_transformers_['cat'].named_steps['onehot']
cat_encoded_cols = list(cat_encoder.get_feature_names_out(categorical_features))
total_pipeline_features = numeric_features + cat_encoded_cols + (infra_cols + ["Is_Rush_Hour"])

print("=== PREPROCESSOR ARTIFACT AUDIT ===")
print(f"Artifact Location:          ../models/preprocessor.joblib")
print(f"Single-record input shape:  {sample_test_row.shape}")
print(f"Single-record output shape: {transformed_mock.shape}")
print(f"Total Transformed Features: {len(total_pipeline_features)}")
print("Smoke Test Status:          SUCCESS (Ready for Stage 6 Modeling & Stage 9 Deployment)")

=== PREPROCESSOR ARTIFACT AUDIT ===
Artifact Location:          ../models/preprocessor.joblib
Single-record input shape:  (1, 23)
Single-record output shape: (1, 92)
Total Transformed Features: 92
Smoke Test Status:          SUCCESS (Ready for Stage 6 Modeling & Stage 9 Deployment)
